# Bronze Incremental Ingestion

> **Incrementally load raw data into the Bronze layer** using watermark-based tracking for safe, repeatable runs.

---

### 📌 What This Notebook Does
- 📥 **Ingests raw data** from source into Bronze Delta tables
- 🕐 **Tracks watermarks** to process only new/changed records
- 🔁 **Idempotent execution** — safe to re-run without duplicates

---


   
## 📦 Step 1: Imports & Setup

Import necessary libraries and configure the environment for incremental bronze ingestion.

| Library | Purpose |
|---|---|
| `pyspark.sql.functions` | Column transformations & expressions |
| `delta.tables.DeltaTable` | Delta Lake merge & upsert operations |
| `datetime` | Watermark timestamp handling |
| `uuid` | Unique batch run identifiers |

---


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog novacartdbx")

In [0]:
spark.sql("create schema if not exists bronze_schema")

# Step 2: Create Bronze Table

Store a **watermark** for each table to make the incremental load safe and repeatable.

The watermark helps the pipeline remember:
- ✅ The **latest timestamp** already processed
- ✅ The **latest primary key processed-at** timestamp
- ✅ The **number of rows** written in the latest run

> 💡 *This is what makes the Bronze incremental load **safe** and **idempotent**.*


In [0]:
spark.sql("""
          create table if not exists novacartdbx.bronze_schema.ingstion_control(
              layer string,
              table_name string,
              ts_col string,
              ps_col string,
              last_successful_ts timestamp,
              last_successful_pk bigint,
              last_run_id string,
              rows_written bigint,
              run_status string,
              updated_at timestamp
          )
          using delta
          """)

   
## 🗂️ Step 3: Source Table Configuration

Define which source table to ingest into Bronze, and specify the key columns for incremental tracking.

| Setting | Purpose |
|---|---|
| **Primary Key** | Uniquely identifies each record for upserts |
| **Timestamp / Watermark Column** | Tracks new or changed records since the last run |
| **`bronze_run_id`** | A unique identifier generated for each pipeline run |

> 💡 *These settings ensure that each incremental load is **traceable**, **repeatable**, and **conflict-free**.*

---

In [0]:
table_config = {
    "orders": {"pk_col" : "order_id", "ts_col" : "updated_at"},
    "products": {"pk_col" : "product_id", "ts_col" : "updated_at"},
    "payments":{"pk_col" : "payment_id", "ts_col" : "processed_at"}
}

bronze_run_id = str(uuid.uuid4())
print("Current Bronze Run ID:", bronze_run_id)

   
   
## 🛠️ Step 4: Helper Functions

Reusable functions that keep the incremental load logic **clean**, **readable**, and **maintainable**.

| Function | Purpose |
|---|---|
| `get_last_successful_watermark()` | Reads the last processed watermark from the control table |
| `upsert_bronze_control()` | Updates the Bronze control table after a successful load |

> 💡 *These helpers abstract away bookkeeping so the main pipeline code stays focused on **what** to load, not **how** to track it.*

---

In [0]:
def get_last_successful_watermark(table_name: str):
    ctrl = (spark.table("novacartdbx.bronze_schema.ingstion_control")
            .filter((F.col("layer")=="bronze") &
            (F.col("table_name")==table_name) &
            (F.col("run_status")=="success"))
            .orderBy(F.col("updated_at").desc())
            .limit(1)
        )
            
    rows = ctrl.collect()      
    if not rows:
        return None, None
    return rows[0].last_successful_ts, rows[0].last_successful_pk    

In [0]:
def upsert_bronze_control(table_name,ts_col,pk_col,last_ts,last_pk,rows_written,run_id):
    control_df = spark.createDataFrame(
        [
            (
                "bronze",
                table_name,
                ts_col,
                pk_col,
                last_ts,
                int(last_pk) if last_pk is not None else None,
                run_id,
                int(rows_written),
                "success",
                datetime.utcnow()
            )
        ],
        schema="""
        layer string,
        table_name string,
        ts_col string,
        ps_col string,
        last_successful_ts timestamp,
        last_successful_pk bigint,
        last_run_id string,
        rows_written bigint,
        run_status string,
        updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark, "novacartdbx.bronze_schema.ingstion_control")
    (
        dt.alias("t")
        .merge(control_df.alias("s"), "t.table_name = s.table_name AND t.layer = s.layer")
        .whenMatchedUpdate(set={
            "ts_col": "s.ts_col",
            "ps_col": "s.ps_col",
            "last_successful_ts": "s.last_successful_ts",
            "last_successful_pk": "s.last_successful_pk",
            "last_run_id": "s.last_run_id",
            "rows_written": "s.rows_written",
            "run_status": "s.run_status",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
     )
    

   
   
## 🔄 Step 5: Bronze Incremental Load Loop

Main ingestion logic — iterates over each configured source table and performs an **incremental load** into the Bronze layer.

| Step | Action |
|------|--------|
| 1️⃣ | **Read last watermark** from the control table |
| 2️⃣ | **Read source SQL table** from the upstream system |
| 3️⃣ | **Filter** only new or changed rows since the last watermark |
| 4️⃣ | **Add Bronze audit columns** (run ID, ingestion timestamp) |
| 5️⃣ | **Append** the filtered rows into the Bronze Delta table |
| 6️⃣ | **Update** the control table with the new watermark |

> 💡 *Each iteration is self-contained — if one table fails, the others are unaffected.*

---

In [0]:
for table_name, cfg in table_config.items():
    pk_col = cfg["pk_col"]
    ts_col = cfg["ts_col"]
    source_table = f"dbxnovacart_catalog.dbo.{table_name}"
    target_table = f"novacartdbx.bronze_schema.{table_name}_raw"
    last_successful_ts, last_successful_pk = get_last_successful_watermark(table_name)
    print(f"\n === Processing {table_name} ===")
    print(f"\n Last successful ts: {last_successful_ts}") 
    print(f"last successful pk: {last_successful_pk}")
    
    source_df = spark.read.table(source_table).withColumn(ts_col,F.col(ts_col).cast("timestamp"))

    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        rows_to_load = source_df.filter(
            (F.col(ts_col) > F.lit(last_successful_ts)) |
            (
                (F.col(ts_col) == F.lit(last_successful_ts)) &
                (F.col(pk_col).cast("long") > F.lit(int(last_successful_pk)))
            )
        )

    rows_to_load = (
        rows_to_load.withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("bronze_run_id", F.lit(bronze_run_id))
        .withColumn("bronze_source_name", F.lit(source_table))
    )        

    rows_count = rows_to_load.count()

    print(f"{table_name} rows_to_load = {rows_count}")

    if rows_count == 0:
        print(f"no news rows for {table_name}.")
        upsert_bronze_control(
            table_name,
            ts_col,
            pk_col,
            last_successful_ts,
            last_successful_pk,
            rows_count,
            bronze_run_id
        )
        continue

    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)

    max_ts = rows_to_load.agg(F.max(ts_col).alias("max_ts")).collect()[0]["max_ts"]
    max_pk = (
        rows_to_load.filter(F.col(ts_col) == F.lit(max_ts))
        .agg(F.max(F.col(pk_col).cast("long")).alias("max_pk"))
        .collect()[0]["max_pk"]
        )
    

    upsert_bronze_control(table_name,ts_col,pk_col,max_ts,max_pk,rows_count,bronze_run_id)
    print(f"Wrote {rows_count} rows to {target_table}")

    
   

   
## ✅ Step 6: Quick Validation

Final check — print the **Bronze row counts** and display the **control table** to verify that the incremental logic is working correctly.

| Check | What to Look For |
|-------|-------------------|
| 🔢 **Row Counts** | Each Bronze table should reflect the expected number of ingested rows |
| 📋 **Control Table** | Watermarks, run IDs, and statuses should be up to date |
| 🔁 **Re-run Safety** | Running again with no new data should show `rows_written = 0` |

> 💡 *If everything looks correct, your Bronze incremental pipeline is working end to end!*

---

In [0]:
print("order bronze count : ", spark.sql("select count(*) from novacartdbx.bronze_schema.orders_raw").collect()[0][0])
print("product bronze count : ", spark.sql("select count(*) from novacartdbx.bronze_schema.products_raw").collect()[0][0])
print("product bronze count : ", spark.sql("select count(*) from novacartdbx.bronze_schema.payments_raw").collect()[0][0])


display(spark.sql("select * from novacartdbx.bronze_schema.ingstion_control").orderBy("table_name"))
